<a href="https://colab.research.google.com/github/jawad66108/codealpha_tasks/blob/main/Mahi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U unsloth
!pip install -q "datasets>=3.4.1,<4.4.0" "transformers>=4.51.3,<=5.5.0" "trl>=0.18.2,<=0.24.0"
!pip install -q accelerate bitsandbytes peft

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Model loaded and LoRA adapters attached")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/unsloth/mistral-7b-instruct-v0.3-bnb-4bit/resolve/main/chat_template.jinja
Retrying in 1s [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/unsloth/mistral-7b-instruct-v0.3-bnb-4bit/resolve/main/chat_template.jinja
Retrying in 2s [Retry 2/5].
Unsloth 2026.9.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model loaded and LoRA adapters attached


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="mahnoor_dataset.jsonl", split="train")

def formatting_prompts_func(examples):
    texts = []
    for convo in examples["conversations"]:
        text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)

print(dataset[0]["text"][:500])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/6381 [00:00<?, ? examples/s]

<s>[INST] Her: Aoa
Jawad yeh new num ha mera dusra del kr dain
Mahnoor Saleem
Me: Ok
Eman Ali ok add kar tho[/INST] Ok
Movie start sy 00:27 to 00:59 tk mujy clip chahiye mein ny record Kiya ha but woh background music ki awaz hi nhi aa rhi....!?</s>


In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/6381 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,381 | Num Epochs = 3 | Total steps = 2,394
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,4.612950
20,3.764422
30,3.449508
40,3.175683
50,3.088571
60,3.126817
70,3.034755
80,3.027319
90,2.925694
100,2.915206


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-500.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-1000.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-1500.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2000/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-2000.


Step,Training Loss
10,4.612950
20,3.764422
30,3.449508
40,3.175683
50,3.088571
60,3.126817
70,3.034755
80,3.027319
90,2.925694
100,2.915206


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2394/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-2394.


In [ ]:
FastLanguageModel.for_inference(model)

def chat_with_model(message, history=""):
    prompt = f"{history}[INST] {message} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Try a few test messages
test_messages = [
    "Kya kar rahi ho",
    "Assalamualaikum",
    "Aj kaisa din tha",
]

for msg in test_messages:
    print(f"You: {msg}")
    print(f"Response: {chat_with_model(msg)}")
    print()

Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


You: Kya kar rahi ho


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: Kuch nhi

You: Assalamualaikum


Both `max_new_tokens` (=100) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Response: Walikum Assalam 
Kasy hain

You: Aj kaisa din tha
Response: Bohot hi masruf hain Jawad Bhai 🔪
Achaa g🌚



In [ ]:
model.save_pretrained("mahnoor_lora")
tokenizer.save_pretrained("mahnoor_lora")
print("LoRA adapters saved")

Unsloth: Restored added_tokens_decoder metadata in mahnoor_lora/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in mahnoor_lora.


LoRA adapters saved


In [ ]:
model.save_pretrained_merged("mahnoor_merged", tokenizer, save_method="merged_16bit")
print("Merged model saved")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in mahnoor_merged/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in mahnoor_merged.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...



Unsloth: Preparing safetensor model files:  67%|██████▋   | 2/3 [03:10<01:44, 104.15s/it]

model-00003-of-00003.safetensors: reconstructing file:   0%|          |  0.00B / 4.55GB            

model-00003-of-00003.safetensors: downloading bytes:           |  0.00B            



Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [05:11<00:00, 103.94s/it]


Unsloth: Merge process complete. Saved to `/content/mahnoor_merged`
Merged model saved


In [ ]:
model.save_pretrained_gguf("mahnoor_merged", tokenizer, quantization_method="q4_k_m")
print("GGUF export complete")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00003.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.




Unsloth: Preparing safetensor model files: 100%|██████████| 3/3 [00:00<00:00, 7033.49it/s]


Unsloth: Merging weights into 16bit:   0%|          | 0/3 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  33%|███▎      | 1/3 [01:29<02:59, 89.51s/it]

Unsloth: Merging weights into 16bit:  67%|██████▋   | 2/3 [03:19<01:41, 101.65s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [04:33<00:00, 91.27s/it]


Unsloth: Merge process complete. Saved to `/content/mahnoor_merged`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10798-mix-659e406 (app-b10798-mix-659e406-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['mahnoor_merged_gguf/mistral-7b-instruct-v0.3.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions complet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q -U unsloth
!pip install -q "datasets>=3.4.1,<4.4.0" "transformers>=4.51.3,<=5.5.0" "trl>=0.18.2,<=0.24.0"
!pip install -q accelerate bitsandbytes peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Model loaded and LoRA adapters attached")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Model loaded and LoRA adapters attached


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="mahnoor_dataset.jsonl", split="train")

def formatting_prompts_func(examples):
    texts = []
    for convo in examples["conversations"]:
        text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"][:300])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/6381 [00:00<?, ? examples/s]

<s>[INST] Her: Aoa
Jawad yeh new num ha mera dusra del kr dain
Mahnoor Saleem
Me: Ok
Eman Ali ok add kar tho[/INST] Ok
Movie start sy 00:27 to 00:59 tk mujy clip chahiye mein ny record Kiya ha but woh background music ki awaz hi nhi aa rhi....!?</s>


In [4]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "/content/drive/MyDrive/mahnoor_checkpoints",
        save_strategy = "steps",
        save_steps = 200,
        save_total_limit = 3,
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/6381 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,381 | Num Epochs = 3 | Total steps = 2,394
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,4.613267
20,3.764013
30,3.449697
40,3.176140
50,3.125652
60,3.136302
70,3.046135
80,3.033634
90,2.933151
100,2.920744


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-800/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-800.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-1000/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-1000.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mahnoor_checkpoints/checkp

Step,Training Loss
10,4.613267
20,3.764013
30,3.449697
40,3.176140
50,3.125652
60,3.136302
70,3.046135
80,3.033634
90,2.933151
100,2.920744


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-1400/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-1400.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-1600/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-1600.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-1800/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-1800.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-2000/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/mahnoor_checkpoints/check

In [1]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

conversation = []

def chat(user_msg):
    conversation.append({"role": "user", "content": user_msg})
    inputs = tokenizer.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, use_cache=True)
    reply = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    conversation.append({"role": "assistant", "content": reply})
    print("Bot:", reply)

chat("kiyah kaar rahi")

ModuleNotFoundError: No module named 'unsloth'

In [6]:
import os
print(os.listdir("/content/drive/MyDrive/mahnoor_checkpoints/checkpoint-2394"))

['README.md', 'adapter_model.safetensors', 'adapter_config.json', 'chat_template.jinja', 'tokenizer_config.json', 'tokenizer.json', 'tokenizer.model', 'training_args.bin', 'optimizer.pt', 'scheduler.pt', 'scaler.pt', 'rng_state.pth', 'trainer_state.json']


In [2]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/mahnoor_checkpoints/checkpoint-2394",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)
# then use the same chat() function as above

ModuleNotFoundError: No module named 'unsloth'

In [3]:
!pip install -q -U unsloth
!pip install -q "datasets>=3.4.1,<4.4.0" "transformers>=4.51.3,<=5.5.0" "trl>=0.18.2,<=0.24.0"
!pip install -q accelerate bitsandbytes peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3

In [3]:
from google.colab import drive
drive.mount('/content/drive')

In [4]:
# 1. Install packages
!pip install -q -U unsloth
!pip install -q "datasets>=3.4.1,<4.4.0" "transformers>=4.51.3,<=5.5.0" "trl>=0.18.2,<=0.24.0"
!pip install -q accelerate bitsandbytes peft

# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 3. Load your trained model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/mahnoor_checkpoints/checkpoint-2394",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

# 4. Set up chat function
conversation = []

def chat(user_msg):
    conversation.append({"role": "user", "content": user_msg})
    inputs = tokenizer.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, use_cache=True)
    reply = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    conversation.append({"role": "assistant", "content": reply})
    print("Bot:", reply)

# 5. Chat!
chat("aoa kiyah kar rahi hein")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-2394 as a legacy tokenizer.
Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: Apko tou kuch pta hi nhi na hota tou mein kiya kru🙃


In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/mahnoor_checkpoints/checkpoint-2394",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

# 4. Set up chat function
conversation = []

def chat(user_msg):
    conversation.append({"role": "user", "content": user_msg})
    inputs = tokenizer.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, use_cache=True)
    reply = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    conversation.append({"role": "assistant", "content": reply})
    print("Bot:", reply)

# 5. Chat!
chat("mein nay bola kiyah ro raha ap pta nahi kiyah bolrahi hein")

==((====))==  Unsloth 2026.9.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-2394 as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: Mein ny kb bola apko pta chl jay ga


In [6]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/mahnoor_checkpoints/checkpoint-2394",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

# 4. Set up chat function
conversation = []

def chat(user_msg):
    conversation.append({"role": "user", "content": user_msg})
    inputs = tokenizer.apply_chat_template(
        conversation, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, use_cache=True)
    reply = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    conversation.append({"role": "assistant", "content": reply})
    print("Bot:", reply)

# 5. Chat!
chat("acha ye batein uni mein din kasa guzra")

==((====))==  Unsloth 2026.9.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load /content/drive/MyDrive/mahnoor_checkpoints/checkpoint-2394 as a legacy tokenizer.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Bot: Ehm
Or apko...🤐
